<a href="https://colab.research.google.com/github/kouzi5816/ML_Engineering_Repo/blob/main/01_MLOps/MLOps_Notebook_with_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

ライブラリのインストール

In [1]:
!pip install mlflow fastapi uvicorn pyngrok -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 41.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 42.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.9/94.9 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.4/94

ライブラリのインポート

In [2]:
import json
import requests
import mlflow
import mlflow.sklearn
import subprocess, time
from pyngrok import ngrok
from google.colab import userdata
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

Machine Learning Modelの学習

In [3]:
# データ準備
X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# パラメータを複数試行
for n_estimators in list([50, 100, 200]):
    # MLflow で実験を記録する
    with mlflow.start_run():
        # パラメータを記録
        mlflow.log_param("n_estimators", n_estimators)

        # モデルを学習
        model = RandomForestClassifier(n_estimators=n_estimators)
        model.fit(X_train, y_train)

        # 精度を記録
        acc = accuracy_score(y_test, model.predict(X_test))
        mlflow.log_metric("accuracy", acc)

        # モデル本体を保存
        mlflow.sklearn.log_model(model, "model")

        print(f"accuracy: {acc:.3f}")

2026/06/30 04:34:25 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/30 04:34:25 INFO mlflow.store.db.utils: Updating database tables
2026/06/30 04:34:28 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


accuracy: 0.933


2026/06/30 04:34:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


accuracy: 0.933


2026/06/30 04:35:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


accuracy: 0.933


学習結果の確認

In [4]:
runs = mlflow.search_runs()
print(runs[["params.n_estimators", "metrics.accuracy"]])
# → 3回分の結果が表として表示される

  params.n_estimators  metrics.accuracy
0                 200          0.933333
1                 100          0.933333
2                  50          0.933333


API用ファイルの作成

In [6]:
api_code = """
from fastapi import FastAPI
from pydantic import BaseModel
from sklearn.datasets import load_iris
from sklearn.ensemble import RandomForestClassifier
import numpy as np

app = FastAPI()

# モデルを起動時に学習（本来は保存済みモデルを読む）
X, y = load_iris(return_X_y=True)
model = RandomForestClassifier(n_estimators=100).fit(X, y)
labels = load_iris().target_names

# 入力の型を定義
class IrisInput(BaseModel):
    sepal_length: float
    sepal_width: float
    petal_length: float
    petal_width: float

# 予測エンドポイント
@app.post("/predict")
def predict(data: IrisInput):
    X_input = [[data.sepal_length, data.sepal_width,
                data.petal_length, data.petal_width]]
    pred = model.predict(X_input)[0]
    prob = model.predict_proba(X_input)[0].max()
    return {
        "prediction": labels[pred],
        "confidence": round(float(prob), 3)
    }

@app.get("/")
def root():
    return {"message": "ML API is running"}
"""

with open("main.py", "w") as f:
    f.write(api_code)
print("main.py を作成しました")

main.py を作成しました


ローカルサーバーの立ち上げとAPI用URL発行

In [7]:
# ngrok の認証トークンの設定
# https://dashboard.ngrok.com/get-started/your-authtoken からアカウント作成と取得
# Google ColabのSecretsに保存したAPIキーを読み込み
API_KEY = userdata.get('API_Key')
ngrok.set_auth_token(API_KEY)

# サーバー起動
proc = subprocess.Popen(["uvicorn", "main:app", "--port", "8000"])
time.sleep(2)

# 外からアクセスできるURLを発行
url = ngrok.connect(8000)
print(f"API URL: {url}")
print(f"ドキュメント: {url}/docs")
# → https://xxxx.ngrok.io のようなURLが表示される

API URL: NgrokTunnel: "https://phony-prison-augmented.ngrok-free.dev" -> "http://localhost:8000"
ドキュメント: NgrokTunnel: "https://phony-prison-augmented.ngrok-free.dev" -> "http://localhost:8000"/docs


APIの利用

In [8]:
response = requests.post(
    f"{url.public_url}/predict",
    json={
        "sepal_length": 2.1,
        "sepal_width":  0.5,
        "petal_length": 0.4,
        "petal_width":  1.5
    }
)
print(response.json())

{'prediction': 'versicolor', 'confidence': 0.48}
